In [ ]:
library(future)
library(maftools)
library(dplyr)
library(data.table)
library(gdata)
plan("multicore", workers=15)
options(future.globals.maxSize=120000 * 1024^2)

# merge maf object
- Discovery/Validation: pre-filter for depth of >= 50, max_af < 0.01, filter == PASS
- Circulating: pre-filter for depth of >= 10, max_af < 0.01, filter == PASS

## data loading 

In [ ]:
folder <- "Discovery"
folder <- "Validation"
folder <- "Circulating"

In [ ]:
main_path <- paste0("/expanse/lustre/projects/csd728/m7huang/placental_genomics_RNAseq/somatic_filtered_20251216/",
                    folder, "/")

sampleNames <- as.list(read.table(paste0(main_path, folder, "_sample_list.txt"))$V1)
mafs <- paste0(main_path, "maf_samples/", sampleNames, ".maf")

filepath <- file.path(paste0(main_path, "maf_objs"))

if (!dir.exists(filepath)) {
  dir.create(filepath)
  cat(paste("Created folder:", filepath))
} else {
  cat(paste("Folder already exists:", filepath))
}
print(length(mafs))

## maf filtering

In [ ]:
# filters maf files and stores into maf_objs directory

filterMafObj <- function(maf, sampleName){
    mafObj <- read.maf(maf)
    
    rare <- subsetMaf(mafObj, query = ("t_depth >= 10"), mafObj = FALSE)
    # rare <- subsetMaf(mafObj, query = ("t_depth >= 50"), mafObj = FALSE)

    # rare2 <- subset(rare, (!Variant_Classification %in% c("RNA", "Silent","3'UTR", "Intron", "Splice_Region", "5'UTR","3'Flank","5'Flank")))
    rare3 <- subset(rare, (MAX_AF < 0.01 | is.na(MAX_AF) | MAX_AF=="" | MAX_AF=="-" | MAX_AF=="."))
    rare4 <- subset(rare3, FILTER == "PASS")
    rare5 <- read.maf(rare4)
    
    fileName <- paste(sampleName, ".RData", sep = "")
    save(rare4, file = file.path(filepath, fileName))
}
mapply(filterMafObj, mafs, sampleNames)

## maf merging

In [ ]:
# loads all maf objects under unique names for upcoming merge
readMafObj <- function(maf, sampleName){
    load(maf, envir = .GlobalEnv)
    mv(from = "rare4", to = sampleName, envir = .GlobalEnv)
}

maf_files <- list.files(filepath, full.names = TRUE)
mapply(readMafObj, maf_files, sampleNames)

In [ ]:
ls()

In [ ]:
mafObjs <- mget(ls(pattern = "^Sample"))

# mafObjs <- mget(ls(pattern = "^P"))
merged_maf_obj <- merge_mafs(mafObjs)
merged_maf_obj

In [ ]:
save(merged_maf_obj, file = paste0(main_path, folder, "_merged_maf_20251219.RData"))

# rare and rare damaging variants

In [ ]:
variant_filtration <- function(all_variants){
    rare <- subset(all_variants@data, 
               (!Variant_Classification %in% c("RNA", "Silent", "Splice_Site")))
    print(paste0("All variants: ", nrow(rare)))

    rare_gnomad <- subset(rare, ((!is.na(MAX_AF) | !MAX_AF=="" | !MAX_AF=="-" | !MAX_AF==".")))
    print(paste0("gnomAD: ", nrow(rare_gnomad)))

    rare_novel <- subset(rare, ((is.na(MAX_AF) | MAX_AF=="" | MAX_AF=="-" | MAX_AF==".") & !(is.na(dbSNP_RS) | dbSNP_RS == "novel"|dbSNP_RS == "")))
    print(paste0("Rare novel: ", nrow(rare_novel)))

    rare_indels <- subset(rare, !(Variant_Type == "SNP") & (!(is.na(dbSNP_RS) | dbSNP_RS == "novel"|dbSNP_RS == "") | !(is.na(MAX_AF) | MAX_AF=="" | MAX_AF=="-" | MAX_AF==".")))
    print(paste0("Rare indels: ", nrow(rare_indels)))

    rare_merge1 <- rbind(rare_gnomad, rare_novel)
    rare_merge <- rbind(rare_merge1, rare_indels)
    print(paste0("Total rare: ", nrow(rare_merge)))


    rare_unique <- rare_merge %>% distinct()
    print(paste0("Total unique rare: ", nrow(rare_unique)))

    rare_unique_maf <- read.maf(rare_unique)
    rare_unique_maf
}

In [ ]:
# remove variants occurring in all patients (or some >% of patients) to remove false positives

patient_filtration <- function(rare_unique){
    df <- rare_unique
    # get number of samples
    n_Source_MAF <- df %>% pull(Tumor_Sample_Barcode) %>% unique() %>% length()
    
    rare_vars <- df %>% 
        select(Tumor_Sample_Barcode, dbSNP_RS) %>% 
        group_by(dbSNP_RS) %>% 
        count() %>% # count how many samples the variant is present in
        ungroup() %>% 
        filter(n<(0.7*n_Source_MAF)) %>% # filter out variants present in 70%+ of samples
        pull(dbSNP_RS)

    rare_filtered <- subset(rare_unique, dbSNP_RS %in% rare_vars)
    print(paste0("Rare variants (> 70%): ", nrow(rare_filtered)))

    read.maf(rare_filtered)
}

In [ ]:
gene_filtration <- function(rare_filtered){
    flags <- c('TTN','MUC16','OBSCN','AHNAK2','SYNE1','FLG','MUC5B','DNAH17','PLEC',
          'DST','SYNE2','NEB','HSPG2','LAMA5','AHNAK','HMCN1', 'USH2A','DNAH11',
          'MACF1','MUC17','DNAH5','GPR98','FAT1','PKD1','MDN1','RNF213','RYR1',
          'DNAH2','DNAH3','DNAH8','DNAH1','DNAH9', 'ABCA13','APOB','SRRM2','CUBN',
          'SPTBN5','PKHD1','LRP2','FBN3','CDH23','DNAH10','FAT4','RYR3','PKHD1L1',
          'FAT2','CSMD1','PCNT', 'COL6A3','FRAS1','FCGBP','DNAH7','RP1L1','PCLO',
          'ZFHX3','COL7A1','LRP1B','FAT3','EPPK1','VPS13C','HRNR','MKI67','MYO15A',
          'STAB1','ZAN','UBR4','VPS13B','LAMA1','XIRP2','BSN','KMT2C','ALMS1','CELSR1',
          'TG','LAMA3','DYNC2H1','KMT2D','BRCA2','CMYA5','SACS', 'STAB2','AKAP13',
          'UTRN','VWF','VPS13D','ANK3','FREM2','PKD1L1','LAMA2','ABCA7','LRP1','ASPM',
          'MYOM2','PDE4DIP','TACC2','MUC2','TEP1','HELZ2','HERC2','ABCA4')
    rare_filtered_no_flags <- subset(rare_filtered, !Hugo_Symbol %in% flags)
    rare_filtered_no_flags_maf <- read.maf(rare_filtered_no_flags)
    
    filter_genes <- getGeneSummary(rare_filtered_no_flags_maf)[,c('Hugo_Symbol', 'MutatedSamples')]
    num_samples <- rare_filtered_no_flags_maf@data %>% 
                        pull(Tumor_Sample_Barcode) %>% 
                        unique() %>% length() 
    print(num_samples)

    filter_genes <- subset(filter_genes, MutatedSamples > num_samples/2)$Hugo_Symbol
    print(filter_genes)   

    rare_final <- subset(rare_filtered_no_flags, !Hugo_Symbol %in% filter_genes)
    read.maf(rare_final)
}

In [ ]:
rare_damaging_id <- function(maf_pass){
    rare_damaging <- subset(maf_pass, !((is.na(CLIN_SIG) | CLIN_SIG == "" |CLIN_SIG == ".") & 
                                       (is.na(SIFT) | SIFT == ""| SIFT == ".") & 
                                       (is.na(PolyPhen) | PolyPhen == ""|PolyPhen == ".")) &
                                    ((!CLIN_SIG %like% "benign") | (is.na(CLIN_SIG) | CLIN_SIG == "" |CLIN_SIG == ".")) &  
                                        ((SIFT %like% "deleterious") | (is.na(SIFT) | SIFT == ""| SIFT == ".")) &
                                        ((PolyPhen %like% "damaging") | (is.na(PolyPhen) | PolyPhen == "" |PolyPhen == ".")) |
                                    (Variant_Classification %in% c("Nonsense_Mutation","Nonstop_Mutation")))
    rare_damaging
}                        

In [ ]:
folder <- "Discovery"
main_path <- paste0("/expanse/lustre/projects/csd728/m7huang/placental_genomics_RNAseq/somatic_filtered_20251216/",
                    folder, "/")
load(paste0(main_path, folder, "_merged_maf_20251219.RData"))

disc_var <- variant_filtration(merged_maf_obj)
plotmafSummary(maf = disc_var, rmOutlier = TRUE, 
               addStat = 'median', dashboard = TRUE, titvRaw = TRUE)
disc_var

disc_var <- patient_filtration(disc_var@data)
plotmafSummary(maf = disc_var, rmOutlier = TRUE, 
               addStat = 'median', dashboard = TRUE, titvRaw = TRUE)

disc_var <- gene_filtration(disc_var@data)
plotmafSummary(maf = disc_var, rmOutlier = TRUE, 
               addStat = 'median', dashboard = TRUE, titvRaw = TRUE)
write.mafSummary(disc_var, 
                 basename = paste0(main_path, folder, "_rare_final_maf"))

maf.titv = titv(disc_var, plot = FALSE, useSyn = TRUE)
#plot titv summary
plotTiTv(res = maf.titv)
write.csv(maf.titv, paste0(main_path, folder, "_titv.csv"))

rare_damage <- read.maf(rare_damaging_id(disc_var@data))
plotmafSummary(maf = rare_damage, rmOutlier = TRUE, 
               addStat = 'median', dashboard = TRUE, titvRaw = TRUE)
write.mafSummary(rare_damage, 
                 basename = paste0(main_path, folder, "_rare_damaging_only_final_maf"))

In [ ]:
folder <- "Validation"
main_path <- paste0("/expanse/lustre/projects/csd728/m7huang/placental_genomics_RNAseq/somatic_filtered_20251216/",
                    folder, "/")
load(paste0(main_path, folder, "_merged_maf_20251219.RData"))

val_var <- variant_filtration(merged_maf_obj)
plotmafSummary(maf = val_var, rmOutlier = TRUE, 
               addStat = 'median', dashboard = TRUE, titvRaw = TRUE)
val_var

val_var <- patient_filtration(val_var@data)
plotmafSummary(maf = val_var, rmOutlier = TRUE, 
               addStat = 'median', dashboard = TRUE, titvRaw = TRUE)
val_var

val_var <- gene_filtration(val_var@data)
plotmafSummary(maf = val_var, rmOutlier = TRUE, 
               addStat = 'median', dashboard = TRUE, titvRaw = TRUE)
val_var
write.mafSummary(val_var, 
                 basename = paste0(main_path, folder, "_rare_final_maf"))

maf.titv = titv(val_var, plot = FALSE, useSyn = TRUE)
#plot titv summary
plotTiTv(res = maf.titv)
write.csv(maf.titv, paste0(main_path, folder, "_titv.csv"))

rare_damage <- read.maf(rare_damaging_id(val_var@data))
plotmafSummary(maf = rare_damage, rmOutlier = TRUE, 
               addStat = 'median', dashboard = TRUE, titvRaw = TRUE)
write.mafSummary(rare_damage, 
                 basename = paste0(main_path, folder, "_rare_damaging_only_final_maf"))

In [ ]:
folder <- "Circulating"
main_path <- paste0("/expanse/lustre/projects/csd728/m7huang/placental_genomics_RNAseq/somatic_filtered_20251216/",
                    folder, "/")
load(paste0(main_path, folder, "_merged_maf_20251219.RData"))

In [ ]:
batches <- c("Discovery", "Validation_1", "Validation_2")

for (batch in batches){
    print(batch)
    meta <- read.csv("Circulating/final_metadata_20251228.csv")
    sub <- subset(meta, Batch == batch)
    sub <- as.character(paste0("Sample", sub$Run))
    print(length(sub))

    circ_subset <- subsetMaf(maf = merged_maf_obj, tsb = sub)
    plotmafSummary(maf = circ_subset, rmOutlier = TRUE, 
                   addStat = 'median', dashboard = TRUE, titvRaw = TRUE)
    circ_subset
    
    circ_var <- variant_filtration(circ_subset)
    plotmafSummary(maf = circ_var, rmOutlier = TRUE, 
                   addStat = 'median', dashboard = TRUE, titvRaw = TRUE)
    circ_var
    
    circ_var <- patient_filtration(circ_var@data)
    plotmafSummary(maf = circ_var, rmOutlier = TRUE, 
                   addStat = 'median', dashboard = TRUE, titvRaw = TRUE)
    circ_var
    
    circ_var <- gene_filtration(circ_var@data)
    plotmafSummary(maf = circ_var, rmOutlier = TRUE, 
                   addStat = 'median', dashboard = TRUE, titvRaw = TRUE)
    circ_var
    write.mafSummary(circ_var, 
                     basename = paste0(main_path, folder, "_", batch, "_rare_final_maf"))

    maf.titv = titv(circ_var, plot = FALSE, useSyn = TRUE)
    #plot titv summary
    plotTiTv(res = maf.titv)
    write.csv(maf.titv, paste0(main_path, folder, "_", batch, "_titv.csv"))

    rare_damage <- read.maf(rare_damaging_id(circ_var@data))
    plotmafSummary(maf = rare_damage, rmOutlier = TRUE, 
                   addStat = 'median', dashboard = TRUE, titvRaw = TRUE)
    write.mafSummary(rare_damage, 
                     basename = paste0(main_path, folder, "_", batch, "_rare_damaging_only_final_maf"))

}

split for final sample level vcfs

# Visualization - ED 2A

In [ ]:
library(maftools)
library(tidyr)
library(dplyr)

library(ggbreak) 
library(ggplot2)

In [ ]:
rare <- read.maf("Discovery/Discovery_rare_final_maf_maftools.maf")@data

rare_counts <- as.data.frame(table(rare$Consequence))
rare_counts <- rare_counts %>%
    separate_rows(Var1, sep = ",") %>%
    group_by(Var1) %>%
    summarise(Freq = sum(Freq), .groups = "drop") %>%
    as.data.frame()

rare_counts

In [ ]:
recurrent_genes <- as.data.frame(table(unique(rare[,c("Hugo_Symbol", "Tumor_Sample_Barcode")])$Hugo_Symbol))

In [ ]:
rare_counts <- rbind(rare_counts, data.frame(
  Var1 = c("total", "recurrent genes", "unique genes"),
  Freq = c(nrow(rare), nrow(subset(recurrent_genes, Freq > 1)), length(unique(rare$Hugo_Symbol)))
))

rare_counts

In [ ]:
rare_damaging <- read.maf("Discovery/Discovery_rare_damaging_only_final_maf_maftools.maf")@data

rare_damaging_counts <- as.data.frame(table(rare_damaging$Consequence))
rare_damaging_counts <- rare_damaging_counts %>%
    separate_rows(Var1, sep = ",") %>%
    group_by(Var1) %>%
    summarise(Freq = sum(Freq), .groups = "drop") %>%
    as.data.frame()

rare_damaging_counts

In [ ]:
recurrent_genes <- as.data.frame(table(unique(rare_damaging[,c("Hugo_Symbol", "Tumor_Sample_Barcode")])$Hugo_Symbol))

In [ ]:
rare_damaging_counts <- rbind(rare_damaging_counts, data.frame(
  Var1 = c("total", "recurrent genes", "unique genes"),
  Freq = c(nrow(rare_damaging), nrow(subset(recurrent_genes, Freq > 1)), length(unique(rare_damaging$Hugo_Symbol)))
))

rare_damaging_counts

In [ ]:
rare_counts <- rare_counts %>%
  mutate(variant = "rare somatic")

rare_damaging_counts <- rare_damaging_counts %>%
  mutate(variant = "rare damaging somatic")

df_combined <- bind_rows(rare_counts, rare_damaging_counts)


In [ ]:
library(dplyr)

# Add variant labels
rare_counts <- rare_counts %>%
  mutate(variant = "rare somatic")

rare_damaging_counts <- rare_damaging_counts %>%
  mutate(variant = "rare damaging somatic")

# Combine
df_combined <- bind_rows(rare_counts, rare_damaging_counts)

# Ensure all Variant.Class x variant combinations exist
df_combined_complete <- expand.grid(
  Var1 = unique(df_combined$Var1),
  variant = unique(df_combined$variant),
  stringsAsFactors = FALSE
) %>%
  left_join(df_combined, by = c("Var1", "variant")) %>%
  mutate(Count = ifelse(is.na(Freq), 0, Freq)) %>%
  select(Var1, Count, variant)


In [ ]:
df_combined_complete

In [ ]:
var_sum <- df_combined_complete %>%
  rename(
    Variant.Class = Var1,
    Count = Count,
    Category = variant
  )
var_sum$Category <- factor(var_sum$Category, levels = c("rare somatic", "rare damaging somatic"))
var_sum$Variant.Class <- factor(var_sum$Variant.Class, 
                                levels = c("recurrent genes", "unique genes",
                                          "stop_gained", "stop_lost", "inframe_insertion",
                                          "inframe_deletion", "frameshift_variant",
                                          "splice_region_variant", "missense_variant", "total"))


In [ ]:
var_sum

In [ ]:
ggplot(var_sum, aes(fill = Category, y = Count, x = Variant.Class)) +
  geom_bar(
    position = "dodge",
    stat = "identity",
    alpha = 9/10
  ) +
  scale_fill_manual(values = c( "#76B7BD","#447273")) +
  scale_y_break(c(250, 600), scales = 1) + 
  scale_y_continuous(limits = c(0, 4000)) +   # set upper limit

  theme_bw(base_family = "Arial") +
  theme(
    axis.text.x = element_text(
      face = "plain",
      color = "Black",
      size = 11,
      angle = 90
    ),
    axis.text.y = element_text(
      face = "plain",
      color = "Black",
      size = 12,
      angle = 0
    ),
    legend.position = "top",
    legend.title = element_blank(),
    legend.text = element_text(size = 12, face = "plain"),
    axis.title.x = element_blank(),
    axis.title.y = element_blank(), 
    panel.grid.major = element_blank(),  # remove major grid lines
    panel.grid.minor = element_blank(),  # remove minor grid lines
    axis.ticks.y.right = element_blank(),
    axis.text.y.right = element_blank()
  )


In [ ]:
pdf("Variant_class_summary_barplot_somatic_20260106.pdf")
ggplot(var_sum, aes(fill = Category, y = Count, x = Variant.Class)) +
  geom_bar(
    position = "dodge",
    stat = "identity",
    alpha = 9/10
  ) +
  scale_fill_manual(values = c( "#76B7BD","#447273")) +
  scale_y_break(c(250, 600), scales = 1) + 
  scale_y_continuous(limits = c(0, 4000)) +   # set upper limit

  theme_bw() +
  theme(
    axis.text.x = element_text(
      face = "plain",
      color = "Black",
      size = 11,
      angle = 90
    ),
    axis.text.y = element_text(
      face = "plain",
      color = "Black",
      size = 12,
      angle = 0
    ),
    legend.position = "top",
    legend.title = element_blank(),
    legend.text = element_text(size = 12, face = "plain"),
    axis.title.x = element_blank(),
    axis.title.y = element_blank(), 
    panel.grid.major = element_blank(),  # remove major grid lines
    panel.grid.minor = element_blank(),  # remove minor grid lines
    axis.ticks.y.right = element_blank(),
    axis.text.y.right = element_blank()
  )

dev.off()